## Tool definition

In [6]:
from pathlib import Path
import sys

from dotenv import load_dotenv

def find_project_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "minimax_model.py").exists():
            return path
    raise FileNotFoundError("Could not find minimax_model.py")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)

True

In [1]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

d:\langchain\lca-lc-foundations\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [4]:
tool1.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [7]:
from langchain.agents import create_agent
from minimax_model import get_model

agent = create_agent(
    model=get_model(),
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [8]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

<think>
The user asked: "What is the square root of 467?" We used the square_root function and got approximately 21.61018278497431. We should respond with the result. Usually, we may include units, rounding, explanation. But likely just answer the numeric result.

We should produce a friendly answer.

Given that we have the result, we can say: The square root of 467 is approximately 21.61.

Better to give more precision? The result has many digits. We can present to maybe 3 decimal places: 21.610.

We could also note that 467 is not a perfect square, so the root is irrational.

Thus answer: sqrt(467) ≈ 21.61018278.

Proceed.
</think>

The square root of 467 is approximately **21.6102** (more precisely, 21.61018278497431).


In [9]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='890c75a4-c663-4289-badf-1d6141e64848'),
 AIMessage(content='<think>\nWe need to calculate the square root of 467. Use the function square_root.\n</think>\n\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 192, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'total_characters': 0}, 'model_provider': 'openai', 'model_name': 'MiniMax-M2.7', 'system_fingerprint': None, 'id': '064b784c05b519a121a2a03567f886f3', 'finish_reason': 'tool_calls', 'logprobs': None}, name='MiniMax AI', id='lc_run--019e016e-af89-7922-b261-a202516a04b9-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_function_803ccfbf6d53_1', 'type': 'tool_call'}], invalid_tool_calls=[],

In [10]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_function_803ccfbf6d53_1', 'type': 'tool_call'}]
